# Phase 4 CJPE Stage 3 with the V3 model
## BitFit adaptation of the trained V3 checkpoint

This third model-by-model stage adapts the existing Phase 1 V3 checkpoint to binary case-outcome classification. V3 is the encoder-only Prefix-LM: it uses an 11-layer, 768-wide Transformer with 12 heads, learned positional embeddings, RMSNorm, GELU feed-forward blocks, and a 512-token bidirectional prefix inside its 1,024-token pretraining sequence. We strict-load its trained weights, keep the bidirectional prefix encoder, and train only the original feed-forward bias vectors plus a new two-class head. The Phase 1 language-model training is not repeated.

**Research question:** Can sparse bias-only BitFit adaptation make the V3 legal-language representation useful for case-outcome prediction, while leaving its large weight matrices frozen?

V1 Stage 1 used additive LoRA and V2 Stage 2 used LoRMA+. V3 Stage 3 uses a different adaptation family, BitFit, so the project can compare low-rank updates with sparse bias updates. Because backbone and method differ at the same time, the score comparison is a practical model-run comparison, not a controlled estimate of the method effect. This is one stage of the overall experiment plan; it is not the complete submission set, and it does not perform knowledge distillation.

The notebook saves the best development checkpoint, training history, class-wise and context-ablation metrics, test predictions, and a ZIP with root-level `prediction.json`. It makes no claim about final quality before training and official scoring.

References: Ben Zaken et al., [BitFit (ACL 2022)](https://aclanthology.org/2022.acl-short.1/); V3 source: `v3.ipynb`, cell 26.

**Run order:** attach the listed Kaggle inputs, restart the session, then use **Run All** from the first cell. The data/tokenizer cell depends on `ASSET`, created in “Inputs and run settings”; running later cells alone after a restart leaves that variable undefined.


## 1. Inputs and run settings

Attach `train.json`, `dev.json`, `test.json`, `v3_model.pt`, and `bpe_en_25k.json` to Kaggle exactly once each. Select a GPU accelerator. Input lookup stops if an asset is missing or duplicated.

In [ ]:
import time

import os,json,math,time,gc,zipfile,random,copy
from pathlib import Path
from collections import Counter
from functools import lru_cache
import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torch.utils.checkpoint import checkpoint
SEED=781; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'; AMP_DTYPE=torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
INPUT_ROOT=Path('/kaggle/input'); OUT=Path('/kaggle/working/phase4_v3_bitfit'); OUT.mkdir(parents=True,exist_ok=True)
# Long-document strategy: overlapping 512-token windows, spaced across the case.
CHUNK_TOKENS=512; CHUNK_STRIDE=384; MAX_CHUNKS_PER_CASE=8; TRAIN_CHUNKS_PER_CASE=1
BATCH_CASES=8; GRAD_ACCUM_STEPS=2; INFER_CHUNK_BATCH=16
# BitFit and task-head optimizer settings.
BITFIT_LR=1e-4; HEAD_DROPOUT=0.10; HEAD_LR=3e-4; WEIGHT_DECAY=0.01; MAX_EPOCHS=4; PATIENCE=2; PROGRESS_EVERY=50
WARMUP_RATIO=0.03; MAX_GRAD_NORM=1.0

def find_one(name):
    found=sorted(p for p in INPUT_ROOT.rglob(name) if p.is_file())
    if len(found)!=1: raise FileNotFoundError(f'Expected one {name}; found {len(found)}: {found[:8]}')
    return found[0]
ASSET={n:find_one(n) for n in ('train.json','dev.json','test.json','v3_model.pt','bpe_en_25k.json')}
print('Device:',DEVICE,'GPU:',torch.cuda.get_device_name(0) if USE_AMP else 'none')
for n,p in ASSET.items(): print(n,'->',p)


## 2. Split checks and the V3 tokenizer

Training and development labels must be integer 0/1. Test rows are reduced to `id` and `text`; any extra test fields are discarded. The original V3 Phase 1 BPE-25k tokenizer and exact token IDs are used with a bounded cache for repeated words.

**Duplicate-record handling:** the supplied train/dev files contain exact repeated case rows. The loader keeps one copy per ID, removes only rows with identical text and label, and raises an error if an ID has conflicting content or labels.


In [ ]:

def load_cases(path,labelled):
    with Path(path).open(encoding='utf-8') as f: rows=json.load(f)
    if not isinstance(rows,list): raise TypeError(f'{path}: expected JSON list')
    result=[]; seen={}; duplicate_count=0
    for i,r in enumerate(rows):
        if not isinstance(r,dict) or 'id' not in r or 'text' not in r: raise ValueError(f'{path}: row {i} lacks id/text')
        cid=str(r['id']); text=r['text']
        if not isinstance(text,str): raise TypeError(f'{cid}: text must be string')
        label=r.get('label') if labelled else None
        if labelled and (type(label) is not int or label not in (0,1)): raise ValueError(f'{cid}: label must be integer 0/1')
        normalized={'id':cid,'text':text}
        if labelled: normalized['label']=label
        if cid in seen:
            if result[seen[cid]] != normalized:
                raise ValueError(f'{path}: conflicting duplicate id {cid} (text or label differs)')
            duplicate_count+=1
            continue
        r.clear(); r.update(normalized)
        seen[cid]=len(result); result.append(r)
    if duplicate_count: print(f'{path}: removed {duplicate_count} exact duplicate rows; kept {len(result)} unique IDs')
    return result
TRAIN_ROWS=load_cases(ASSET['train.json'],True); DEV_ROWS=load_cases(ASSET['dev.json'],True); TEST_ROWS=load_cases(ASSET['test.json'],False)
sets=[{r['id'] for r in rows} for rows in (TRAIN_ROWS,DEV_ROWS,TEST_ROWS)]
if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)): raise ValueError('Split IDs overlap')
print('Split sizes:',*[len(x) for x in (TRAIN_ROWS,DEV_ROWS,TEST_ROWS)])
print('Train label counts:',dict(Counter(r['label'] for r in TRAIN_ROWS)))
print('Development label counts:',dict(Counter(r['label'] for r in DEV_ROWS)))
print('Test loader retains id/text only.')

with ASSET['bpe_en_25k.json'].open(encoding='utf-8') as f: bpe=json.load(f)
bpe_itos=bpe['itos']; bpe_merges=bpe['merges']; bpe_stoi={t:i for i,t in enumerate(bpe_itos)}
if bpe['vocab_size']!=25000 or len(bpe_itos)!=25000: raise ValueError('Expected V1 BPE-25k')
merge_to_id={}
for i,(left,right) in enumerate(bpe_merges): merge_to_id[(bpe_stoi[left],bpe_stoi[right])]=261+i
PAD_TOKEN_ID=bpe_stoi['<pad>']; BOS_TOKEN_ID=bpe_stoi['<s>']; EOS_TOKEN_ID=bpe_stoi['</s>']

class ByteLevelBPE25k:
    def __init__(self,itos,stoi,merge_to_id,merges):
        self.itos=itos; self.stoi=stoi; self.merge_to_id=merge_to_id; self.merges=merges; self.vocab_size=len(itos)
        self.pad_token_id=stoi['<pad>']; self.bos_token_id=stoi['<s>']; self.eos_token_id=stoi['</s>']
        self.unk_token_id=stoi['<unk>']; self.word_boundary_id=stoi['<wb>']
    @lru_cache(maxsize=100000)
    def _encode_word(self,word):
        seq=[4+b for b in word.encode('utf-8')]+[self.word_boundary_id]
        while len(seq)>1:
            best_pos=None; best_id=None
            for i in range(len(seq)-1):
                new_id=self.merge_to_id.get((seq[i],seq[i+1]))
                if new_id is not None and (best_id is None or new_id<best_id): best_id=new_id; best_pos=i
            if best_pos is None: break
            seq=seq[:best_pos]+[best_id]+seq[best_pos+2:]
        return seq
    def encode(self,text,add_bos=False,add_eos=False):
        ids=[self.bos_token_id] if add_bos else []
        for word in text.split(): ids.extend(self._encode_word(word))
        if add_eos: ids.append(self.eos_token_id)
        return ids
    def _to_bytes(self,tid):
        tok=self.itos[tid]
        if tok.startswith('<0x') and tok.endswith('>'): return bytes([int(tok[3:-1],16)])
        if tok=='<wb>': return b' '
        if tok in {'<pad>','<unk>','<s>','</s>'}: return b''
        if tok.startswith('<bpe_'):
            j=tid-261
            if 0<=j<len(self.merges):
                a,b=self.merges[j]; return self._to_bytes(self.stoi[a])+self._to_bytes(self.stoi[b])
        return b''
    def decode(self,ids):
        out=bytearray()
        for tid in ids:
            if 0<=int(tid)<self.vocab_size: out.extend(self._to_bytes(int(tid)))
        return bytes(out).decode('utf-8',errors='replace')

tokenizer=ByteLevelBPE25k(bpe_itos,bpe_stoi,merge_to_id,bpe_merges)
probe='The Supreme Court of India has jurisdiction over constitutional matters.'
decoded_probe=tokenizer.decode(tokenizer.encode(probe))
# The final <wb> token decodes to one trailing space by design.
assert decoded_probe.rstrip()==probe, repr(decoded_probe)
print('V1 tokenizer loaded; vocabulary:',tokenizer.vocab_size)


## 3. Rebuild the exact V3 architecture and load its trained weights

This cell embeds the model definition from `v3.ipynb`, cell 26. The complete V3 model is reconstructed and the checkpoint is loaded strictly before any classifier or adaptation is added. A missing or mismatched state-dictionary key stops the run.

In [ ]:
# 9. V3 Encoder-Only Prefix-LM
#
# V3 uses a single encoder-style Transformer stack.
#
# Attention pattern:
#   Prefix tokens  -> bidirectional attention
#   Target tokens  -> prefix + previous target tokens
#
# This allows the model to perform autoregressive
# continuation without a separate decoder or cross-attention.
#
# V3 uses:
# - Encoder-only Prefix-LM
# - Standard Multi-Head Attention
# - Learned positional embeddings
# - RMSNorm
# - Prefix-bidirectional + target-causal attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation
# - Tied token embeddings and LM head
#
# This is intentionally different from both V1 and V2.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):

    def __init__(
        self,
        dim,
        eps=1e-6
    ):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Prefix-LM Attention
# ---------------------------------------------------------

class V3MultiHeadAttention(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.dropout = dropout

    def _split_heads(self, x):

        B, T, C = x.shape

        x = x.view(
            B,
            T,
            self.n_head,
            self.head_dim
        )

        return x.transpose(1, 2)

    def _merge_heads(self, x):

        B, H, T, D = x.shape

        x = x.transpose(
            1,
            2
        ).contiguous()

        return x.view(
            B,
            T,
            H * D
        )

    def forward(
        self,
        x,
        attention_mask=None,
        past_key_value=None,
        use_cache=False
    ):

        q = self._split_heads(
            self.q_proj(x)
        )

        k = self._split_heads(
            self.k_proj(x)
        )

        v = self._split_heads(
            self.v_proj(x)
        )

        if past_key_value is not None:

            past_k, past_v = past_key_value

            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = self._merge_heads(y)

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Prefix-LM Attention Mask
# ---------------------------------------------------------

def create_prefix_lm_mask(
    sequence_length,
    prefix_length,
    device
):
    """
    Creates the attention pattern for Prefix-LM.

    Prefix:
        Every prefix token can attend to every
        other prefix token.

    Target:
        Every target token can attend to:
        - all prefix tokens
        - itself
        - previous target tokens

    Future target tokens are masked.
    """

    mask = torch.zeros(
        sequence_length,
        sequence_length,
        device=device,
        dtype=torch.bool
    )

    # Prefix tokens attend bidirectionally
    mask[
        :prefix_length,
        :prefix_length
    ] = True

    # Target tokens attend to all prefix tokens
    mask[
        prefix_length:,
        :prefix_length
    ] = True

    # Target tokens attend causally to target tokens
    target_length = (
        sequence_length - prefix_length
    )

    causal_target_mask = torch.tril(
        torch.ones(
            target_length,
            target_length,
            device=device,
            dtype=torch.bool
        )
    )

    mask[
        prefix_length:,
        prefix_length:
    ] = causal_target_mask

    return mask


# ---------------------------------------------------------
# GELU Feed Forward Network
# ---------------------------------------------------------

class V3FeedForward(nn.Module):

    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        x = self.fc1(x)

        x = F.gelu(x)

        x = self.fc2(x)

        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# V3 Transformer Block
# ---------------------------------------------------------

class V3TransformerBlock(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = RMSNorm(
            n_embd
        )

        self.self_attention = V3MultiHeadAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm2 = RMSNorm(
            n_embd
        )

        self.ffn = V3FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        attention_mask=None,
        past_key_value=None,
        use_cache=False
    ):

        # Pre-normalized self-attention.
        normalized_x = self.norm1(x)

        attention_output, present = (
            self.self_attention(
                normalized_x,
                attention_mask=attention_mask,
                past_key_value=past_key_value,
                use_cache=use_cache
            )
        )

        x = x + attention_output

        # Pre-normalized feed-forward network.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V3 Encoder-Only Prefix-LM
# ---------------------------------------------------------

class V3LegalLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layers,
        n_head,
        max_sequence_length,
        prefix_length,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.max_sequence_length = (
            max_sequence_length
        )

        self.prefix_length = (
            prefix_length
        )

        # Shared token embedding.
        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Learned positional embeddings.
        self.position_embedding = nn.Embedding(
            max_sequence_length,
            n_embd
        )

        # Single encoder-style Transformer stack.
        self.layers = nn.ModuleList([
            V3TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Tied language-model head.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        labels=None,
        past_key_values=None,
        use_cache=False
    ):

        B, T = input_ids.shape

        # Determine the position offset during
        # autoregressive generation.
        if past_key_values is None:

            position_offset = 0

        else:

            position_offset = (
                past_key_values[0][0].size(-2)
            )

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=input_ids.device
        )

        x = self.token_embedding(
            input_ids
        )

        x = x + self.position_embedding(
            positions
        ).unsqueeze(0)

        # During normal training we process the
        # complete prefix + target sequence.
        #
        # During generation, the prefix is first
        # processed with its bidirectional mask.
        # Afterwards the generated tokens use the
        # cached keys and values.

        attention_mask = None

        if past_key_values is None:

            attention_mask = create_prefix_lm_mask(
                sequence_length=T,
                prefix_length=min(
                    self.prefix_length,
                    T
                ),
                device=input_ids.device
            )

        presents = [] if use_cache else None

        for layer_index, layer in enumerate(
            self.layers
        ):

            past = None

            if past_key_values is not None:

                past = past_key_values[
                    layer_index
                ]

            x, present = layer(
                x,
                attention_mask=attention_mask,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:

                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        loss = None

        if labels is not None:

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                labels.reshape(-1),
                ignore_index=-100
            )

        return (
            logits,
            loss,
            presents
        )


print("V3 Encoder-Only Prefix-LM model classes defined.") 

In [ ]:
# V3 Phase 1 architecture and checkpoint metadata from the original V3 notebook.
N_EMBD=768; N_LAYERS=11; N_HEAD=12
MODEL_CONTEXT_LENGTH=1024; SOURCE_LENGTH=512; TARGET_LENGTH=512
FFN_MULTIPLIER=4; DROPOUT=0.0
V3_CONFIG={'architecture':'encoder_only_prefix_lm','vocab_size':tokenizer.vocab_size,
            'n_embd':N_EMBD,'n_layers':N_LAYERS,'n_head':N_HEAD,
            'source_length':SOURCE_LENGTH,'target_length':TARGET_LENGTH,
            'ffn_multiplier':FFN_MULTIPLIER,'dropout':DROPOUT,
            'position_embedding':'learned','normalization':'rmsnorm',
            'attention':'standard_multi_head',
            'attention_pattern':'bidirectional_prefix_causal_target',
            'kv_cache':True,'tied_embeddings':True}
v3=V3LegalLM(vocab_size=tokenizer.vocab_size,n_embd=N_EMBD,n_layers=N_LAYERS,
             n_head=N_HEAD,max_sequence_length=MODEL_CONTEXT_LENGTH,
             prefix_length=SOURCE_LENGTH,ffn_multiplier=FFN_MULTIPLIER,dropout=DROPOUT)
ckpt=torch.load(ASSET['v3_model.pt'],map_location='cpu',weights_only=False)
if ckpt.get('config') != V3_CONFIG: raise ValueError(f'V3 checkpoint config mismatch: {ckpt.get("config")}')
state=ckpt.get('model_state_dict',ckpt)
missing,unexpected=v3.load_state_dict(state,strict=True)
assert not missing and not unexpected
PRETRAINED_TOTAL_PARAMS=sum(p.numel() for p in v3.parameters())
print('Strict V3 checkpoint load passed; unique model parameters:',f'{PRETRAINED_TOTAL_PARAMS:,}')
print('Phase 1 best validation loss:',ckpt.get('best_val_loss'),'perplexity:',ckpt.get('best_val_perplexity'),'step:',ckpt.get('best_step'))
v3.to(DEVICE).eval()


## 4. BitFit from scratch as a controlled, sparse adaptation

BitFit updates bias terms in a pretrained Transformer while leaving its matrix weights frozen [1]. The V3 attention query/key/value/output projections have no bias parameters, and RMSNorm has a learned scale but no bias. The V3 feed-forward layers do have biases, so this notebook makes those 22 vectors trainable across the 11 blocks. It also trains a new two-class head, which is required because the Phase 1 language-model head predicts vocabulary tokens rather than court outcomes.

The classification path uses the V3 prefix exactly as a bidirectional source representation: each case window is at most 512 tokens, equal to V3's trained prefix length. We add a boolean key-padding mask to prevent right-padding tokens from influencing real-token states. The mask changes no trained weights. A masked mean over real-token states gives the window representation, and logits from selected document windows are averaged into the case prediction.

This is deliberately a small, interpretable update. It can cheaply test whether shifts in feed-forward biases are enough to expose task-relevant patterns in the frozen V3 representation. The trade-off is limited capacity: BitFit may under-adapt if case-outcome prediction requires changing directions in the frozen weight matrices.

In [ ]:
# BitFit: train only pretrained feed-forward biases plus the new task head.
for p in v3.parameters(): p.requires_grad_(False)
for name,p in v3.named_parameters():
    if name.startswith('layers.') and name.endswith('.bias'):
        p.requires_grad_(True)

class CaseClassifier(nn.Module):
    def __init__(self,backbone):
        super().__init__(); self.backbone=backbone
        self.head=nn.Sequential(nn.Dropout(HEAD_DROPOUT),nn.Linear(N_EMBD,2))
        for p in self.head.parameters(): p.requires_grad_(True)
    def forward(self,input_ids,attention_mask):
        mask=attention_mask.bool()
        b,t=input_ids.shape
        positions=torch.arange(t,device=input_ids.device).unsqueeze(0)
        x=self.backbone.token_embedding(input_ids)+self.backbone.position_embedding(positions)
        # In this classifier, every position is in the V3 bidirectional prefix.
        # SDPA boolean True means that the key is allowed to be attended to.
        allowed_keys=mask[:,None,None,:]
        for layer in self.backbone.layers:
            x,_=layer(x,attention_mask=allowed_keys,use_cache=False)
        x=self.backbone.final_norm(x)
        weights=mask.unsqueeze(-1).to(x.dtype)
        pooled=(x*weights).sum(dim=1)/weights.sum(dim=1).clamp_min(1.0)
        return self.head(pooled)

model=CaseClassifier(v3).to(DEVICE)
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
active_total=sum(p.numel() for p in model.parameters())
bitfit_count=sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and n.startswith('backbone.layers.') and n.endswith('.bias'))
print(f'BitFit bias parameters: {bitfit_count:,}; head parameters: {trainable-bitfit_count:,}; total trainable: {trainable:,}; V3 unique total: {PRETRAINED_TOTAL_PARAMS:,}')
assert bitfit_count==42240, f'Unexpected V3 bias count: {bitfit_count}'
assert trainable==43778, f'Unexpected BitFit+head count: {trainable}'


## 5. Long judgments as spaced, overlapping windows

The V3 model learned a 512-token bidirectional prefix, so each case is split into overlapping 512-token windows with 128-token overlap. We select up to eight windows spread from beginning to end. During training, one window is rotated per case each epoch to control compute. Development and test average logits across the selected windows. This samples different regions but does not process every token of every very long judgment. The beginning/middle/end diagnostic measures how sensitive the current run is to document region.

In [ ]:

def select_windows(text):
    ids=tokenizer.encode(text,add_bos=True)
    if not ids: ids=[BOS_TOKEN_ID]
    if len(ids)<=CHUNK_TOKENS: starts=[0]
    else:
        all_starts=list(range(0,len(ids)-CHUNK_TOKENS+1,CHUNK_STRIDE))
        if all_starts[-1]!=len(ids)-CHUNK_TOKENS: all_starts.append(len(ids)-CHUNK_TOKENS)
        take=min(MAX_CHUNKS_PER_CASE,len(all_starts))
        starts=sorted(set(np.linspace(0,len(all_starts)-1,take).round().astype(int).tolist()))
        starts=[all_starts[i] for i in starts]
    chunks=[]
    for start in starts:
        x=ids[start:start+CHUNK_TOKENS]
        chunks.append(x+[PAD_TOKEN_ID]*(CHUNK_TOKENS-len(x)))
    return np.asarray(chunks,dtype=np.int32)

def encode_rows(rows,name):
    counts=[]; chunks=[]
    for i,row in enumerate(rows):
        c=select_windows(row['text']); chunks.append(c); counts.append(len(c))
        if (i+1)%5000==0: print(name,i+1,'/',len(rows))
    print(name,'windows:',sum(counts),'documents:',len(rows),'median windows:',int(np.median(counts)))
    return chunks
TRAIN_WINDOWS=encode_rows(TRAIN_ROWS,'train'); DEV_WINDOWS=encode_rows(DEV_ROWS,'dev'); TEST_WINDOWS=encode_rows(TEST_ROWS,'test')

class TrainDataset(Dataset):
    def __init__(self,rows,windows): self.rows=rows; self.windows=windows; self.epoch=0
    def set_epoch(self,e): self.epoch=e
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        ws=self.windows[i]; idx=(self.epoch+i)%len(ws)
        x=ws[idx]; mask=(x!=PAD_TOKEN_ID)
        return torch.tensor(x,dtype=torch.long),torch.tensor(mask,dtype=torch.bool),self.rows[i]['label']
class EvalDataset(Dataset):
    def __init__(self,rows,windows): self.rows=rows; self.windows=windows
    def __len__(self): return len(self.rows)
    def __getitem__(self,i): return self.windows[i],self.rows[i]['id'],self.rows[i].get('label')

train_ds=TrainDataset(TRAIN_ROWS,TRAIN_WINDOWS)
dev_ds=EvalDataset(DEV_ROWS,DEV_WINDOWS); test_ds=EvalDataset(TEST_ROWS,TEST_WINDOWS)
train_loader=DataLoader(train_ds,batch_size=BATCH_CASES,shuffle=True,num_workers=0,pin_memory=USE_AMP)


## 6. Train the V3 feed-forward biases and task head

The V3 checkpoint is frozen except for its existing feed-forward biases; the new classification head is fully trainable. Class weights come only from training-label counts. We use AdamW with separate learning rates for the pretrained bias vectors and the new head, gradient accumulation, early stopping on development macro-F1, and threshold calibration on development data only. Accuracy is reported as secondary context because the training classes are imbalanced.

In [ ]:

if DEVICE.type!='cuda': raise RuntimeError('Select a Kaggle GPU accelerator before training the V3 checkpoint.')
def scores(y_true,y_pred):
    out={}; f1s=[]
    for c in (0,1):
        tp=sum(a==c and b==c for a,b in zip(y_true,y_pred)); fp=sum(a!=c and b==c for a,b in zip(y_true,y_pred)); fn=sum(a==c and b!=c for a,b in zip(y_true,y_pred))
        p=tp/(tp+fp) if tp+fp else 0.; r=tp/(tp+fn) if tp+fn else 0.; f=2*p*r/(p+r) if p+r else 0.
        out[str(c)]={'precision':p,'recall':r,'f1':f,'support':sum(a==c for a in y_true)}; f1s.append(f)
    out['macro_f1']=sum(f1s)/2; out['accuracy']=sum(a==b for a,b in zip(y_true,y_pred))/max(1,len(y_true))
    return out

@torch.no_grad()
def predict_windows(rows,windows,mode='all',threshold=0.5):
    model.eval(); selected=[]; owners=[]
    for i,ws in enumerate(windows):
        if mode=='begin': chosen=ws[:1]
        elif mode=='middle': chosen=ws[(len(ws)-1)//2:(len(ws)-1)//2+1]
        elif mode=='end': chosen=ws[-1:]
        else: chosen=ws
        selected.extend(chosen); owners.extend([i]*len(chosen))
    flat=np.stack(selected,axis=0)
    if flat.ndim!=2: raise ValueError(f"Expected prediction windows with shape [windows, tokens], got {flat.shape}")
    all_logits=[]
    infer_started=time.time(); infer_total=math.ceil(len(flat)/INFER_CHUNK_BATCH)
    for batch_idx,j in enumerate(range(0,len(flat),INFER_CHUNK_BATCH),start=1):
        ids=torch.as_tensor(flat[j:j+INFER_CHUNK_BATCH],dtype=torch.long,device=DEVICE); mask=ids.ne(PAD_TOKEN_ID)
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=USE_AMP): z=model(ids,mask)
        all_logits.append(z.float().cpu().numpy())
        if batch_idx==1 or batch_idx%PROGRESS_EVERY==0 or batch_idx==infer_total:
            elapsed=time.time()-infer_started; rate=batch_idx/max(elapsed,1e-6); eta=(infer_total-batch_idx)/max(rate,1e-6)
            print(f"Predict {mode}: batch {batch_idx}/{infer_total} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min",flush=True)
    window_logits=np.concatenate(all_logits,axis=0); logits_sum=np.zeros((len(rows),2),dtype=np.float64); counts=np.zeros(len(rows),dtype=np.int64)
    np.add.at(logits_sum,np.asarray(owners),window_logits); np.add.at(counts,np.asarray(owners),1)
    logits=(logits_sum/counts[:,None]).astype(np.float32); probs=torch.softmax(torch.tensor(logits),dim=-1).numpy()[:,1]
    return logits,probs,(probs>=threshold).astype(int).tolist()

counts=np.bincount([r['label'] for r in TRAIN_ROWS],minlength=2)
class_weights=torch.tensor(len(TRAIN_ROWS)/(2*np.maximum(counts,1)),dtype=torch.float,device=DEVICE)
criterion=nn.CrossEntropyLoss(weight=class_weights)
bitfit_params=[p for n,p in model.named_parameters() if p.requires_grad and n.startswith('backbone.layers.') and n.endswith('.bias')]
head_params=[p for n,p in model.named_parameters() if p.requires_grad and n.startswith('head.')]
opt=torch.optim.AdamW([{'params':bitfit_params,'lr':BITFIT_LR},{'params':head_params,'lr':HEAD_LR}],weight_decay=WEIGHT_DECAY)
total_updates=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)*MAX_EPOCHS; warmup=max(1,int(total_updates*WARMUP_RATIO))
def lr_scale(step): return min((step+1)/warmup,1.) if step<warmup else max(0.,(total_updates-step)/(total_updates-warmup))
scheduler=torch.optim.lr_scheduler.LambdaLR(opt,lr_scale)
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and AMP_DTYPE==torch.float16)
history=[]; best_f1=-1.; best_loss=float('inf'); best_epoch=0; stale=0; update=0
for epoch in range(MAX_EPOCHS):
    train_ds.set_epoch(epoch); model.train(); opt.zero_grad(set_to_none=True); loss_sum=0.; seen=0; pending=0; epoch_started=time.time()
    print(f"Starting epoch {epoch+1}/{MAX_EPOCHS}: {len(train_loader)} batches",flush=True)
    for step,(ids,mask,y) in enumerate(train_loader):
        ids=ids.to(DEVICE,non_blocking=True); mask=mask.to(DEVICE,non_blocking=True); y=torch.as_tensor(y,device=DEVICE,dtype=torch.long)
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=USE_AMP): logits=model(ids,mask); loss=criterion(logits,y)
        scaler.scale(loss/GRAD_ACCUM_STEPS).backward(); pending+=1; loss_sum+=float(loss.detach())*len(y); seen+=len(y)
        if step==0 or (step+1)%PROGRESS_EVERY==0 or step+1==len(train_loader):
            elapsed=time.time()-epoch_started; rate=(step+1)/max(elapsed,1e-6); eta=(len(train_loader)-step-1)/max(rate,1e-6)
            print(f"Train epoch {epoch+1}/{MAX_EPOCHS}: batch {step+1}/{len(train_loader)} | running loss {loss_sum/max(seen,1):.4f} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min",flush=True)
        if pending==GRAD_ACCUM_STEPS or step==len(train_loader)-1:
            if pending<GRAD_ACCUM_STEPS:
                correction=GRAD_ACCUM_STEPS/pending
                for p in model.parameters():
                    if p.grad is not None: p.grad.mul_(correction)
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); scheduler.step(); update+=1; pending=0
    dev_logits,dev_probs,dev_pred=predict_windows(DEV_ROWS,DEV_WINDOWS)
    dev_y=[r['label'] for r in DEV_ROWS]; dev_metrics=scores(dev_y,dev_pred)
    record={'epoch':epoch+1,'train_loss':loss_sum/max(seen,1),'dev':dev_metrics,'lr_bitfit':opt.param_groups[0]['lr'],'lr_head':opt.param_groups[1]['lr']}
    history.append(record); print(f"Epoch {epoch+1}/{MAX_EPOCHS} | train loss {record['train_loss']:.4f} | dev macro-F1 {dev_metrics['macro_f1']:.4f} | accuracy {dev_metrics['accuracy']:.4f}")
    if dev_metrics['macro_f1']>best_f1 or (dev_metrics['macro_f1']==best_f1 and record['train_loss']<best_loss):
        best_f1=dev_metrics['macro_f1']; best_loss=record['train_loss']; best_epoch=epoch+1; stale=0
        trainable_names={n for n,p in model.named_parameters() if p.requires_grad}
        torch.save({'trainable_state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items() if k in trainable_names},'epoch':best_epoch,'dev_macro_f1':best_f1,'config':{'method':'BitFit','bitfit_target':'V3 feed-forward biases','chunk_tokens':CHUNK_TOKENS,'max_chunks':MAX_CHUNKS_PER_CASE}},OUT/'v3_bitfit_best.pt')
    else: stale+=1
    (OUT/'training_history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
    if stale>=PATIENCE: print('Early stopping: development macro-F1 did not improve.'); break
print('Best epoch:',best_epoch,'best raw dev macro-F1:',best_f1)


## 7. Restore the best sparse adaptation, inspect context ablations, and calibrate on development only

We report beginning-only, middle-only, end-only, and all-window development metrics. These are diagnostics, not separate submissions. We select the best epoch by raw development macro-F1 at threshold 0.5, then sweep thresholds on development probabilities only. The selected threshold is fixed before test prediction.

In [ ]:
best=torch.load(OUT/'v3_bitfit_best.pt',map_location=DEVICE,weights_only=False)
own=model.state_dict(); own.update(best['trainable_state_dict']); model.load_state_dict(own,strict=True); model.eval()
dev_y=[r['label'] for r in DEV_ROWS]; ablations={}
for mode in ('begin','middle','end','all'):
    _,prob,pred=predict_windows(DEV_ROWS,DEV_WINDOWS,mode=mode)
    ablations[mode]=scores(dev_y,pred)
    print(mode,ablations[mode])
_,dev_prob,dev_pred=predict_windows(DEV_ROWS,DEV_WINDOWS)
thresholds=np.arange(0.30,0.701,0.01); threshold_results=[]
for t in thresholds:
    m=scores(dev_y,(dev_prob>=t).astype(int).tolist()); threshold_results.append((m['macro_f1'],t,m))
_,best_threshold,best_calibrated=max(threshold_results,key=lambda x:(x[0],-abs(x[1]-0.5)))
print('Development-selected threshold:',round(float(best_threshold),2),'metrics:',best_calibrated)
summary={'best_epoch':best_epoch,'best_checkpoint_selection_macro_f1':best_f1,'development_threshold':float(best_threshold),'development_calibrated_metrics':best_calibrated,'context_ablation':ablations,'class_weights':class_weights.cpu().tolist(),'bitfit_parameters':bitfit_count,'trainable_parameters':trainable,'active_model_parameters':active_total,'full_pretrained_parameters':PRETRAINED_TOTAL_PARAMS}
(OUT/'development_metrics.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')


## 8. Predict test cases and create the required submission ZIP

The test rows were loaded using only `id` and `text`; all other fields were discarded before training or prediction. The brief requires one list entry per test case, with integer `pred` in `{0,1}`. The ZIP contains exactly one root-level `prediction.json`.

In [ ]:

test_logits,test_prob,test_pred=predict_windows(TEST_ROWS,TEST_WINDOWS,threshold=float(best_threshold))
prediction=[{'id':row['id'],'pred':int(pred)} for row,pred in zip(TEST_ROWS,test_pred)]
assert len(prediction)==len(TEST_ROWS)
assert [x['id'] for x in prediction]==[x['id'] for x in TEST_ROWS]
assert all(type(x['pred']) is int and x['pred'] in (0,1) for x in prediction)
pred_path=OUT/'prediction.json'; pred_path.write_text(json.dumps(prediction,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path=OUT/'phase4_v3_bitfit_submission.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z: z.write(pred_path,arcname='prediction.json')
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist()==['prediction.json']; assert z.testzip() is None
    roundtrip=json.loads(z.read('prediction.json'))
assert roundtrip==prediction
print('Created:',pred_path,'and',zip_path)
print('Test predictions:',len(prediction),'predicted class counts:',dict(Counter(x['pred'] for x in prediction)))
print('Development metrics:',json.dumps(summary,indent=2))


## What this stage establishes

This notebook evaluates V3 + BitFit as the third model-specific Phase 4 candidate. It does not train the original V3 language model again and does not perform knowledge distillation. The wider experiment plan still needs the distinct experiments required by the Phase 4 brief, including a justified teacher-to-smaller-student distillation run. No score is promised. After Kaggle training, record actual runtime, best epoch, development macro-F1, class-wise precision/recall, context-ablation results, prediction counts, and official score before deciding the next model.